## Country Revenue Summary — country_category_revenue

Purpose: show total EUR revenue by country for Books and Electronics, keeping only the
countries above €40,000.

| Step | What it does | Result |
|------|--------------|--------|
| 1 | Keep only Books and Electronics orders, completed only | relevant orders |
| 2 | Convert each order to EUR (line_total ÷ rate) using its fx_reference_date | amounts in EUR |
| 3 | Sum revenue per country | one total per country |
| 4 | Keep only countries whose total is over €40,000 (HAVING) | small countries dropped |
| 5 | Rank highest revenue first | ordered list |

**Result:**

| country | revenue_eur |
|---------|-------------|
| RO | 146,608.56 |
| HU | 40,823.50 |

Countries below the threshold were excluded: DE (~€38,333) and BG (~€32,254).

### Conclusion
This shows where the Books and Electronics business is strongest. Romania is by far the biggest
market, and Hungary just clears the €40,000 bar. Germany and Bulgaria have real sales too but fall
under the threshold, so they're left out. The €40,000 cut-off does real work here — it keeps the
two meaningful markets and filters out the smaller ones.

Note: the exact figures depend on the exchange rate at run time, so they may shift slightly day to
day; the set of qualifying countries (RO, HU) is stable.


In [1]:
%%sql
-- Total EUR revenue by country, only for Books or Electronics orders, keeping just the countries whose combined revenue is over €40,000, ranked highest first. 
-- Completed orders only (consistent with customer spend — refunds aren't revenue).
-- EUR revenue per country for Books & Electronics, only countries above 40,000, ranked.
-- The project asks for this specific breakdown.
-- same EUR conversion (line_total / rate_to_eur); filter categories; sum per country.
--      HAVING filters AFTER grouping (WHERE filters rows before; HAVING filters the totals).
-- One row per qualifying country, highest revenue first.
CREATE OR REPLACE TABLE country_category_revenue AS
SELECT
  o.country,
  ROUND(SUM(o.line_total / f.rate_to_eur), 2) AS revenue_eur
FROM orders_clean o
JOIN fx_rates f
  ON  f.fx_date  = o.fx_reference_date
  AND f.currency = o.currency
WHERE o.status = 'completed'
  AND o.category IN ('Books', 'Electronics')
GROUP BY o.country
HAVING SUM(o.line_total / f.rate_to_eur) > 40000
ORDER BY revenue_eur DESC

StatementMeta(, ffbd4ec3-c222-48ad-addd-f01f49401ab0, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
%%sql
-- Shows the final country/category revenue table.
-- Only countries over 40,000, ranked highest first.
SELECT * FROM country_category_revenue

StatementMeta(, ffbd4ec3-c222-48ad-addd-f01f49401ab0, 3, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>